In [1]:
import jax.numpy as jnp

from shapiq.game.table import TableGame
from shapiq.players import FinitePlayers

In [2]:
coalitions = jnp.array(
    [
        [0, 0, 0],  # empty
        [1, 0, 0],  # {0}
        [0, 1, 0],  # {1}
        [1, 1, 0],  # {0, 1}
        [0, 0, 1],  # {2}
        [1, 0, 1],  # {0, 2}
        [0, 1, 1],  # {1, 2}
        [1, 1, 1],  # {0, 1, 2}
    ],
    dtype=bool,
)
g = TableGame(jnp.array([[0.0, 1.0, 2.0, 4.0, 3.0, 5.0, 6.0, 10.0]]), FinitePlayers(3))
print("game batch:", g.shape, "shared players:", g.players.shape)
for coalition, value in zip(coalitions.tolist(), g(coalitions).tolist(), strict=True):
    print(coalition, "->", value)

game batch: (1,) shared players: ()
[False, False, False] -> 0.0
[True, False, False] -> 1.0
[False, True, False] -> 2.0
[True, True, False] -> 4.0
[False, False, True] -> 3.0
[True, False, True] -> 5.0
[False, True, True] -> 6.0
[True, True, True] -> 10.0


# Now we build a first MonteCarloApproximator

In [3]:
import jax

In [4]:
from shapiq.sampling._base import SizeSampler
from shapiq.sampling._state import SamplingState

state = SamplingState(g.players, steps=())
sampler = SizeSampler.uniform_over_sizes(
    keys=jax.random.key(0)[None], n_players=g.players.n_players
)

In [5]:
sample, sampler = sampler.sample(state, budget=5)
sample

Sample(coalitions=Array([[[False, False, False],
        [False, False, False],
        [False,  True, False],
        [ True,  True, False],
        [ True,  True,  True]]], dtype=bool), log_weights=Array([[1.3862939, 1.3862939, 2.4849072, 2.4849072, 1.3862939]], dtype=float32), players=FinitePlayers(ids=Array([0, 1, 2], dtype=int32), players=None, latent_n_players=3, n_players=3), unit=1)

In [6]:
new_state = state.append(sample, g(sample.coalitions))
new_state

SamplingState(players=FinitePlayers(ids=Array([0, 1, 2], dtype=int32), players=None, latent_n_players=3, n_players=3), steps=(Evaluations(coalitions=Array([[[False, False, False],
        [False, False, False],
        [False,  True, False],
        [ True,  True, False],
        [ True,  True,  True]]], dtype=bool), log_weights=Array([[1.3862939, 1.3862939, 2.4849072, 2.4849072, 1.3862939]], dtype=float32), players=FinitePlayers(ids=Array([0, 1, 2], dtype=int32), players=None, latent_n_players=3, n_players=3), unit=1, values=Array([[ 0.,  0.,  2.,  4., 10.]], dtype=float32)),))